# DiGiTerra Regression Workflow Notebook

This notebook mirrors the core **Model Preprocessing** and **Modeling** workflow for regression:
- data loading and cleaning
- feature/target selection
- optional outlier handling
- candidate model tuning (Top 3 + MLP)
- multi-seed robustness summary (median + P2.5)
- external validation
- training final model and predicting on new data


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## 1) Configuration

Edit this block for your project.


In [2]:
CONFIG = {
    # Input dataset (xlsx with Literature + Backvalidation, or a csv)
    'input_path': Path('/Users/rowanterra/Desktop/Test_1217/mapping/All_Sites_Test_3_1.xlsx'),

    # Training source
    'train_sheet': 'Literature',
    'train_target': 'TREE',

    # External validation source
    'external_sheet': 'Backvalidation',
    'external_target': 'REEY',  # use equivalent target if naming differs

    # Modeling columns
    'features': ['pH', 'Fe', 'Mn', 'Al', 'SO4'],
    'stratify_col': 'Al',

    # Split and CV
    'test_size': 0.2,
    'cv_folds': 5,
    'single_seed': 42,

    # Robustness seeds
    'seed_start': 0,
    'seed_count': 30,

    # Data cleaning options
    'drop_missing': 'indicatorAndTarget',  # one of: all, indicator, target, indicatorAndTarget
    'drop_zero': 'none',  # one of: none, indicator, target, indicatorAndTarget

    # Candidate pipelines to evaluate
    # outlier_mode one of: none, winsorize_1_99
    'candidates': [
        {'name': 'Ridge', 'outlier_mode': 'winsorize_1_99'},
        {'name': 'ExtraTrees', 'outlier_mode': 'winsorize_1_99'},
        {'name': 'KNN', 'outlier_mode': 'none'},
        {'name': 'MLP', 'outlier_mode': 'winsorize_1_99'},
    ],

    # Grid search toggle
    'use_grid_search': True,
    'grid_n_jobs': -1,
}

# Hyperparameter grids (edit freely)
PARAM_GRIDS = {
    'Ridge': {
        'model__alpha': [0.1, 1.0, 10.0, 30.0],
    },
    'ExtraTrees': {
        'model__n_estimators': [300, 500],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_leaf': [1, 2, 4],
    },
    'KNN': {
        'model__n_neighbors': [5, 7, 9, 11],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2],
    },
    'MLP': {
        'model__hidden_layer_sizes': [(64,), (128, 64), (256, 128)],
        'model__alpha': [1e-5, 1e-4, 1e-3],
        'model__learning_rate_init': [1e-3, 5e-4],
        'model__activation': ['relu', 'tanh'],
    },
}

CONFIG


{'input_path': PosixPath('/Users/rowanterra/Desktop/Test_1217/mapping/All_Sites_Test_3_1.xlsx'),
 'train_sheet': 'Literature',
 'train_target': 'TREE',
 'external_sheet': 'Backvalidation',
 'external_target': 'REEY',
 'features': ['pH', 'Fe', 'Mn', 'Al', 'SO4'],
 'stratify_col': 'Al',
 'test_size': 0.2,
 'cv_folds': 5,
 'single_seed': 42,
 'seed_start': 0,
 'seed_count': 30,
 'drop_missing': 'indicatorAndTarget',
 'drop_zero': 'none',
 'candidates': [{'name': 'Ridge', 'outlier_mode': 'winsorize_1_99'},
  {'name': 'ExtraTrees', 'outlier_mode': 'winsorize_1_99'},
  {'name': 'KNN', 'outlier_mode': 'none'},
  {'name': 'MLP', 'outlier_mode': 'winsorize_1_99'}],
 'use_grid_search': True,
 'grid_n_jobs': -1}

## 2) Utility Functions


In [5]:
class Winsorizer(TransformerMixin, BaseEstimator):
    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(arr, self.lower_q, axis=0)
        self.upper_ = np.nanquantile(arr, self.upper_q, axis=0)
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        return np.clip(arr, self.lower_, self.upper_)


def load_tabular(path: Path, sheet_name: str | None = None) -> pd.DataFrame:
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        if sheet_name is None:
            raise ValueError('sheet_name is required for Excel files')
        return pd.read_excel(path, sheet_name=sheet_name)
    return pd.read_csv(path)


def build_strat_labels(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce')
    for bins in (5, 4, 3, 2):
        try:
            labels = pd.qcut(s, q=bins, labels=False, duplicates='drop')
        except Exception:
            labels = pd.cut(s, bins=bins, labels=False)
        if labels.notna().all():
            vc = labels.value_counts()
            if len(vc) >= 2 and vc.min() >= 2:
                return labels.astype(int)
    raise ValueError('Could not create stable stratification labels.')


def clean_dataframe(df: pd.DataFrame, features: list[str], target: str, drop_missing='indicatorAndTarget', drop_zero='none'):
    cols = list(features) + [target]
    out = df.copy()
    out[cols] = out[cols].apply(pd.to_numeric, errors='coerce')

    if drop_missing == 'all':
        out = out.dropna(how='all')
    elif drop_missing == 'indicator':
        out = out.dropna(subset=features, how='any')
    elif drop_missing == 'target':
        out = out.dropna(subset=[target], how='any')
    elif drop_missing == 'indicatorAndTarget':
        out = out.dropna(subset=cols, how='any')

    if drop_zero == 'indicator':
        out = out[~(out[features] == 0).all(axis=1)]
    elif drop_zero == 'target':
        out = out[out[target] != 0]
    elif drop_zero == 'indicatorAndTarget':
        out = out[~(out[cols] == 0).all(axis=1)]

    return out.reset_index(drop=True)


def get_model(model_name: str, seed: int):
    if model_name == 'Ridge':
        return Ridge(random_state=seed)
    if model_name == 'ExtraTrees':
        return ExtraTreesRegressor(random_state=seed, n_jobs=-1)
    if model_name == 'KNN':
        return KNeighborsRegressor()
    if model_name == 'MLP':
        return MLPRegressor(max_iter=3000, random_state=seed)
    raise ValueError(f'Unsupported model: {model_name}')


def make_pipeline(model_name: str, outlier_mode: str, seed: int):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if outlier_mode == 'winsorize_1_99':
        steps.append(('winsor', Winsorizer(0.01, 0.99)))
    elif outlier_mode != 'none':
        raise ValueError(f'Unsupported outlier_mode: {outlier_mode}')

    if model_name in {'Ridge', 'KNN', 'MLP'}:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', get_model(model_name, seed)))
    return Pipeline(steps)


def evaluate_predictions(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE': mean_absolute_error(y_true, y_pred),
    }


## 3) Load Train and External Data


In [6]:
train_df = load_tabular(CONFIG['input_path'], CONFIG['train_sheet'])
external_df = load_tabular(CONFIG['input_path'], CONFIG['external_sheet'])

print('Train shape:', train_df.shape)
print('External shape:', external_df.shape)

required_train = set(CONFIG['features'] + [CONFIG['train_target'], CONFIG['stratify_col']])
required_ext = set(CONFIG['features'] + [CONFIG['external_target']])
missing_train = [c for c in required_train if c not in train_df.columns]
missing_ext = [c for c in required_ext if c not in external_df.columns]
print('Missing train cols:', missing_train)
print('Missing external cols:', missing_ext)


Train shape: (273, 56)
External shape: (73, 94)
Missing train cols: []
Missing external cols: []


## 4) Preprocess for Internal Split and External Validation


In [7]:
train_clean = clean_dataframe(
    train_df,
    features=CONFIG['features'],
    target=CONFIG['train_target'],
    drop_missing=CONFIG['drop_missing'],
    drop_zero=CONFIG['drop_zero'],
)

external_clean = clean_dataframe(
    external_df,
    features=CONFIG['features'],
    target=CONFIG['external_target'],
    drop_missing=CONFIG['drop_missing'],
    drop_zero=CONFIG['drop_zero'],
)

print('Train clean shape:', train_clean.shape)
print('External clean shape:', external_clean.shape)

X_all = train_clean[CONFIG['features']]
y_all = train_clean[CONFIG['train_target']]
strat_labels = build_strat_labels(train_clean[CONFIG['stratify_col']])

X_external = external_clean[CONFIG['features']]
y_external = external_clean[CONFIG['external_target']]


Train clean shape: (273, 56)
External clean shape: (69, 94)


## 5) Single-Seed Candidate Tuning (Internal Train/Test)


In [8]:
idx = np.arange(len(X_all))
idx_train, idx_test = train_test_split(
    idx,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['single_seed'],
    stratify=strat_labels,
)

X_train, X_test = X_all.iloc[idx_train], X_all.iloc[idx_test]
y_train, y_test = y_all.iloc[idx_train], y_all.iloc[idx_test]

single_seed_rows = []
cv = KFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=CONFIG['single_seed'])

for cand in CONFIG['candidates']:
    model_name = cand['name']
    outlier_mode = cand['outlier_mode']
    pipe = make_pipeline(model_name, outlier_mode, seed=CONFIG['single_seed'])

    if CONFIG['use_grid_search'] and model_name in PARAM_GRIDS:
        search = GridSearchCV(
            estimator=pipe,
            param_grid=PARAM_GRIDS[model_name],
            scoring='r2',
            cv=cv,
            n_jobs=CONFIG['grid_n_jobs'],
        )
        search.fit(X_train, y_train)
        best_estimator = search.best_estimator_
        best_params = search.best_params_
        cv_best = search.best_score_
    else:
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
        pipe.fit(X_train, y_train)
        best_estimator = pipe
        best_params = {}
        cv_best = float(np.mean(cv_scores))

    y_pred = best_estimator.predict(X_test)
    metrics = evaluate_predictions(y_test, y_pred)

    single_seed_rows.append({
        'Model': model_name,
        'Outlier_Mode': outlier_mode,
        'CV_R2': cv_best,
        'Test_R2': metrics['R2'],
        'Test_RMSE': metrics['RMSE'],
        'Test_MAE': metrics['MAE'],
        'Best_Params': best_params,
    })

single_seed_df = pd.DataFrame(single_seed_rows).sort_values('Test_R2', ascending=False).reset_index(drop=True)
single_seed_df


/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/

,Model,Outlier_Mode,CV_R2,Test_R2,Test_RMSE,Test_MAE,Best_Params
0,MLP,winsorize_1_99,0.841270,0.839374,136.253610,61.657082,"{'model__activation': 'relu', 'model__alpha': ..."
1,Ridge,winsorize_1_99,0.833592,0.785868,157.318789,67.085675,{'model__alpha': 30.0}
2,KNN,none,0.810164,0.758611,167.031543,56.394854,"{'model__n_neighbors': 7, 'model__p': 2, 'mode..."
3,ExtraTrees,winsorize_1_99,0.859395,0.622549,208.867052,69.712289,"{'model__max_depth': 10, 'model__min_samples_l..."


## 6) Multi-Seed Robustness (Internal Split)


In [9]:
def run_multiseed_internal(X_all, y_all, strat_labels, candidates, seed_start, seed_count, test_size, cv_folds):
    rows = []
    idx = np.arange(len(X_all))
    for seed in range(seed_start, seed_start + seed_count):
        idx_train, idx_test = train_test_split(
            idx,
            test_size=test_size,
            random_state=seed,
            stratify=strat_labels,
        )
        X_train, X_test = X_all.iloc[idx_train], X_all.iloc[idx_test]
        y_train, y_test = y_all.iloc[idx_train], y_all.iloc[idx_test]
        cv = KFold(n_splits=cv_folds, shuffle=True, random_state=seed)

        for cand in candidates:
            model_name = cand['name']
            outlier_mode = cand['outlier_mode']
            pipe = make_pipeline(model_name, outlier_mode, seed=seed)
            cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)
            metrics = evaluate_predictions(y_test, y_pred)

            rows.append({
                'Seed': seed,
                'Model': model_name,
                'Outlier_Mode': outlier_mode,
                'CV_R2_Mean': float(np.mean(cv_scores)),
                'Test_R2': metrics['R2'],
                'Test_RMSE': metrics['RMSE'],
                'Test_MAE': metrics['MAE'],
            })

    per_seed = pd.DataFrame(rows)
    summary = per_seed.groupby(['Model', 'Outlier_Mode'], as_index=False).agg(
        Seeds=('Seed', 'count'),
        CV_R2_Mean_Median=('CV_R2_Mean', 'median'),
        Test_R2_Median=('Test_R2', 'median'),
        Test_R2_Mean=('Test_R2', 'mean'),
        Test_R2_Std=('Test_R2', 'std'),
        Test_RMSE_Median=('Test_RMSE', 'median'),
        Test_MAE_Median=('Test_MAE', 'median'),
    )

    q = per_seed.groupby(['Model', 'Outlier_Mode'])['Test_R2'].quantile([0.025, 0.975]).unstack().reset_index()
    q.columns = ['Model', 'Outlier_Mode', 'Test_R2_P2_5', 'Test_R2_P97_5']
    summary = summary.merge(q, on=['Model', 'Outlier_Mode'], how='left')
    summary = summary.sort_values(['Test_R2_P2_5', 'Test_R2_Median'], ascending=False).reset_index(drop=True)
    return per_seed, summary

internal_per_seed, internal_summary = run_multiseed_internal(
    X_all=X_all,
    y_all=y_all,
    strat_labels=strat_labels,
    candidates=CONFIG['candidates'],
    seed_start=CONFIG['seed_start'],
    seed_count=CONFIG['seed_count'],
    test_size=CONFIG['test_size'],
    cv_folds=CONFIG['cv_folds'],
)

internal_summary


/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/

,Model,Outlier_Mode,Seeds,CV_R2_Mean_Median,Test_R2_Median,Test_R2_Mean,Test_R2_Std,Test_RMSE_Median,Test_MAE_Median,Test_R2_P2_5,Test_R2_P97_5
0,MLP,winsorize_1_99,30,0.794798,0.896616,0.856610,0.131251,72.907350,37.475636,0.536965,0.954354
1,Ridge,winsorize_1_99,30,0.765136,0.889466,0.829965,0.167381,81.275331,42.395042,0.454897,0.950059
2,KNN,none,30,0.738519,0.837315,0.777450,0.171233,117.655618,40.939564,0.407978,0.913163
3,ExtraTrees,winsorize_1_99,30,0.764891,0.879052,0.807435,0.236472,85.241890,35.931772,0.395445,0.939894


## 7) External Validation (Train on Literature, Test on Backvalidation)


In [10]:
def run_multiseed_external(X_train, y_train, X_external, y_external, candidates, seed_start, seed_count, cv_folds):
    rows = []
    for seed in range(seed_start, seed_start + seed_count):
        cv = KFold(n_splits=cv_folds, shuffle=True, random_state=seed)
        for cand in candidates:
            model_name = cand['name']
            outlier_mode = cand['outlier_mode']
            pipe = make_pipeline(model_name, outlier_mode, seed=seed)

            cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
            pipe.fit(X_train, y_train)
            y_pred_external = pipe.predict(X_external)
            metrics = evaluate_predictions(y_external, y_pred_external)

            rows.append({
                'Seed': seed,
                'Model': model_name,
                'Outlier_Mode': outlier_mode,
                'CV_R2_Mean': float(np.mean(cv_scores)),
                'External_R2': metrics['R2'],
                'External_RMSE': metrics['RMSE'],
                'External_MAE': metrics['MAE'],
            })

    per_seed = pd.DataFrame(rows)
    summary = per_seed.groupby(['Model', 'Outlier_Mode'], as_index=False).agg(
        Seeds=('Seed', 'count'),
        CV_R2_Mean_Median=('CV_R2_Mean', 'median'),
        External_R2_Median=('External_R2', 'median'),
        External_R2_Mean=('External_R2', 'mean'),
        External_R2_Std=('External_R2', 'std'),
        External_RMSE_Median=('External_RMSE', 'median'),
        External_MAE_Median=('External_MAE', 'median'),
    )

    q = per_seed.groupby(['Model', 'Outlier_Mode'])['External_R2'].quantile([0.025, 0.975]).unstack().reset_index()
    q.columns = ['Model', 'Outlier_Mode', 'External_R2_P2_5', 'External_R2_P97_5']
    summary = summary.merge(q, on=['Model', 'Outlier_Mode'], how='left')
    summary = summary.sort_values(['External_R2_P2_5', 'External_R2_Median'], ascending=False).reset_index(drop=True)
    return per_seed, summary

external_per_seed, external_summary = run_multiseed_external(
    X_train=X_all,
    y_train=y_all,
    X_external=X_external,
    y_external=y_external,
    candidates=CONFIG['candidates'],
    seed_start=CONFIG['seed_start'],
    seed_count=CONFIG['seed_count'],
    cv_folds=CONFIG['cv_folds'],
)

external_summary


/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/

,Model,Outlier_Mode,Seeds,CV_R2_Mean_Median,External_R2_Median,External_R2_Mean,External_R2_Std,External_RMSE_Median,External_MAE_Median,External_R2_P2_5,External_R2_P97_5
0,MLP,winsorize_1_99,30,0.832861,0.821656,0.821350,0.010740,66.906897,35.748005,0.802775,0.840457
1,ExtraTrees,winsorize_1_99,30,0.791372,0.712445,0.712207,0.025096,84.957537,41.813757,0.663465,0.751849
2,KNN,none,30,0.760393,0.611146,0.611146,0.000000,98.795039,46.802681,0.611146,0.611146
3,Ridge,winsorize_1_99,30,0.793490,0.584626,0.584626,0.000000,102.108317,57.752952,0.584626,0.584626


## 8) Select Production Candidate (Robustness-First)


In [11]:
selection_table = external_summary.copy().sort_values(['External_R2_P2_5', 'External_R2_Median'], ascending=False).reset_index(drop=True)
selection_table.insert(0, 'Rank', np.arange(1, len(selection_table) + 1))
selection_table


,Rank,Model,Outlier_Mode,Seeds,CV_R2_Mean_Median,External_R2_Median,External_R2_Mean,External_R2_Std,External_RMSE_Median,External_MAE_Median,External_R2_P2_5,External_R2_P97_5
0,1,MLP,winsorize_1_99,30,0.832861,0.821656,0.821350,0.010740,66.906897,35.748005,0.802775,0.840457
1,2,ExtraTrees,winsorize_1_99,30,0.791372,0.712445,0.712207,0.025096,84.957537,41.813757,0.663465,0.751849
2,3,KNN,none,30,0.760393,0.611146,0.611146,0.000000,98.795039,46.802681,0.611146,0.611146
3,4,Ridge,winsorize_1_99,30,0.793490,0.584626,0.584626,0.000000,102.108317,57.752952,0.584626,0.584626


In [12]:
best = selection_table.iloc[0]
BEST_MODEL = best['Model']
BEST_OUTLIER = best['Outlier_Mode']
print('Selected production candidate:')
print({'Model': BEST_MODEL, 'Outlier_Mode': BEST_OUTLIER})


Selected production candidate:
{'Model': 'MLP', 'Outlier_Mode': 'winsorize_1_99'}


## 9) Fit Production Model on Full Training Data


In [13]:
production_pipe = make_pipeline(BEST_MODEL, BEST_OUTLIER, seed=CONFIG['single_seed'])

# Optional: tune final model once more on full training data
if CONFIG['use_grid_search'] and BEST_MODEL in PARAM_GRIDS:
    cv_full = KFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=CONFIG['single_seed'])
    final_search = GridSearchCV(
        estimator=production_pipe,
        param_grid=PARAM_GRIDS[BEST_MODEL],
        scoring='r2',
        cv=cv_full,
        n_jobs=CONFIG['grid_n_jobs'],
    )
    final_search.fit(X_all, y_all)
    production_model = final_search.best_estimator_
    print('Final best params:', final_search.best_params_)
else:
    production_model = production_pipe.fit(X_all, y_all)

external_pred = production_model.predict(X_external)
print('External metrics for production model:', evaluate_predictions(y_external, external_pred))


/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/

Final best params: {'model__activation': 'relu', 'model__alpha': 1e-05, 'model__hidden_layer_sizes': (128, 64), 'model__learning_rate_init': 0.0005}
External metrics for production model: {'R2': 0.8327037613318844, 'RMSE': 64.80144560344547, 'MAE': 33.49901251849411}


## 10) Predict on a New Dataset

Set `NEW_DATA_PATH` to a file containing at least the selected feature columns.


In [14]:
NEW_DATA_PATH = Path('/absolute/path/to/new_dataset.xlsx')
NEW_DATA_SHEET = 'Sheet1'

# Uncomment to run:
# new_df = load_tabular(NEW_DATA_PATH, NEW_DATA_SHEET if NEW_DATA_PATH.suffix.lower() in {'.xlsx', '.xls'} else None)
# missing_new = [c for c in CONFIG['features'] if c not in new_df.columns]
# if missing_new:
#     raise ValueError(f'Missing required feature columns in new data: {missing_new}')
#
# X_new = new_df[CONFIG['features']].apply(pd.to_numeric, errors='coerce')
# y_new_pred = production_model.predict(X_new)
#
# pred_col = f'Predicted_{CONFIG['train_target']}'
# pred_df = new_df.copy()
# pred_df[pred_col] = y_new_pred
#
# out_path = Path('/Users/rowanterra/Desktop/DiGiTerra/docs/new_data_predictions.csv')
# pred_df.to_csv(out_path, index=False)
# print('Saved predictions to', out_path)


## 11) Save Result Tables (Optional)


In [15]:
OUT_DIR = Path('/Users/rowanterra/Desktop/DiGiTerra/docs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

single_seed_df.to_csv(OUT_DIR / 'notebook_single_seed_results.csv', index=False)
internal_per_seed.to_csv(OUT_DIR / 'notebook_internal_per_seed.csv', index=False)
internal_summary.to_csv(OUT_DIR / 'notebook_internal_summary.csv', index=False)
external_per_seed.to_csv(OUT_DIR / 'notebook_external_per_seed.csv', index=False)
external_summary.to_csv(OUT_DIR / 'notebook_external_summary.csv', index=False)
selection_table.to_csv(OUT_DIR / 'notebook_selection_table.csv', index=False)

print('Saved notebook outputs to', OUT_DIR)


Saved notebook outputs to /Users/rowanterra/Desktop/DiGiTerra/docs


## 12) Visual Exports (Pred vs Actual, Residuals, SHAP)

These cells export figures and tables for reporting.


In [20]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    _HAS_SEABORN = True
except Exception:
    _HAS_SEABORN = False

try:
    import shap
    _HAS_SHAP = True
except Exception:
    _HAS_SHAP = False

VIS_DIR = Path('/Users/rowanterra/Desktop/DiGiTerra/docs/notebook_visual_exports')
VIS_DIR.mkdir(parents=True, exist_ok=True)
print('Visual export dir:', VIS_DIR)
print('SHAP available:', _HAS_SHAP)


Visual export dir: /Users/rowanterra/Desktop/DiGiTerra/docs/notebook_visual_exports
SHAP available: True


In [21]:
def export_pred_vs_actual(y_true, y_pred, title, out_path):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(y_true, y_pred, alpha=0.7)
    y_min = float(min(np.min(y_true), np.min(y_pred)))
    y_max = float(max(np.max(y_true), np.max(y_pred)))
    ax.plot([y_min, y_max], [y_min, y_max], linestyle='--')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def export_residual_plot(y_true, y_pred, title, out_path):
    residuals = y_true - y_pred
    fig, ax = plt.subplots(figsize=(7, 5))
    if _HAS_SEABORN:
        sns.histplot(residuals, kde=True, ax=ax)
    else:
        ax.hist(residuals, bins=20, alpha=0.8)
    ax.axvline(0, linestyle='--')
    ax.set_xlabel('Residual (Actual - Predicted)')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def _transform_features_for_model(pipeline, X):
    Xt = X.copy()
    for step_name in ['imputer', 'winsor', 'scaler']:
        if step_name in pipeline.named_steps:
            Xt = pipeline.named_steps[step_name].transform(Xt)
    return Xt


def export_shap_summary_if_available(pipeline, X_ref, feature_names, prefix, max_background=100, max_explain=200):
    if not _HAS_SHAP:
        print(f'Skipping SHAP for {prefix}: shap package not installed.')
        return

    model = pipeline.named_steps['model']
    Xt = _transform_features_for_model(pipeline, X_ref)
    Xt = np.asarray(Xt, dtype=float)
    if Xt.ndim == 1:
        Xt = Xt.reshape(-1, 1)

    n_bg = min(max_background, Xt.shape[0])
    n_explain = min(max_explain, Xt.shape[0])
    bg = Xt[:n_bg]
    Xexp = Xt[:n_explain]

    try:
        if model.__class__.__name__ in {'ExtraTreesRegressor', 'RandomForestRegressor', 'GradientBoostingRegressor'}:
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(Xexp)
        elif model.__class__.__name__ in {'Ridge', 'LinearRegression', 'ElasticNet'}:
            explainer = shap.LinearExplainer(model, bg)
            shap_values = explainer.shap_values(Xexp)
        else:
            f = lambda z: model.predict(z)
            explainer = shap.KernelExplainer(f, bg)
            shap_values = explainer.shap_values(Xexp, nsamples=100)

        plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_values, Xexp, feature_names=feature_names, show=False)
        plt.tight_layout()
        out_png = VIS_DIR / f'{prefix}_shap_summary.png'
        plt.savefig(out_png, dpi=180, bbox_inches='tight')
        plt.close()

        shap_df = pd.DataFrame(Xexp, columns=feature_names)
        shap_arr = np.array(shap_values)
        if shap_arr.ndim == 3:
            shap_arr = shap_arr[0]
        shap_val_df = pd.DataFrame(shap_arr, columns=[f'SHAP_{c}' for c in feature_names])
        shap_out = pd.concat([shap_df.reset_index(drop=True), shap_val_df.reset_index(drop=True)], axis=1)
        shap_out.to_csv(VIS_DIR / f'{prefix}_shap_values.csv', index=False)

        print(f'SHAP exports created for {prefix}')
    except Exception as exc:
        print(f'SHAP export failed for {prefix}: {exc}')


In [22]:
# Build visuals for each candidate using full training and external dataset
visual_rows = []

for cand in CONFIG['candidates']:
    model_name = cand['name']
    outlier_mode = cand['outlier_mode']
    label = f'{model_name}_{outlier_mode}'

    pipe = make_pipeline(model_name, outlier_mode, seed=CONFIG['single_seed'])
    if CONFIG['use_grid_search'] and model_name in PARAM_GRIDS:
        cv_full = KFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=CONFIG['single_seed'])
        search = GridSearchCV(
            estimator=pipe,
            param_grid=PARAM_GRIDS[model_name],
            scoring='r2',
            cv=cv_full,
            n_jobs=CONFIG['grid_n_jobs'],
        )
        search.fit(X_all, y_all)
        model_fit = search.best_estimator_
        best_params = search.best_params_
    else:
        model_fit = pipe.fit(X_all, y_all)
        best_params = {}

    pred_external = model_fit.predict(X_external)
    metrics = evaluate_predictions(y_external, pred_external)

    export_pred_vs_actual(
        y_true=y_external.values,
        y_pred=np.asarray(pred_external),
        title=f'External Pred vs Actual | {label}',
        out_path=VIS_DIR / f'{label}_external_pred_vs_actual.png',
    )

    export_residual_plot(
        y_true=np.asarray(y_external),
        y_pred=np.asarray(pred_external),
        title=f'External Residuals | {label}',
        out_path=VIS_DIR / f'{label}_external_residuals.png',
    )

    export_shap_summary_if_available(
        pipeline=model_fit,
        X_ref=X_all,
        feature_names=CONFIG['features'],
        prefix=f'{label}_train',
    )

    visual_rows.append({
        'Model': model_name,
        'Outlier_Mode': outlier_mode,
        'External_R2': metrics['R2'],
        'External_RMSE': metrics['RMSE'],
        'External_MAE': metrics['MAE'],
        'Best_Params': best_params,
    })

visual_summary = pd.DataFrame(visual_rows).sort_values('External_R2', ascending=False).reset_index(drop=True)
visual_summary.to_csv(VIS_DIR / 'visual_export_model_metrics.csv', index=False)
visual_summary


SHAP exports created for Ridge_winsorize_1_99_train
SHAP exports created for ExtraTrees_winsorize_1_99_train


  0%|          | 0/200 [00:00<?, ?it/s]

SHAP exports created for KNN_none_train


/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP exports created for MLP_winsorize_1_99_train


,Model,Outlier_Mode,External_R2,External_RMSE,External_MAE,Best_Params
0,KNN,none,0.881574,54.521058,30.596358,"{'model__n_neighbors': 7, 'model__p': 1, 'mode..."
1,MLP,winsorize_1_99,0.832704,64.801446,33.499013,"{'model__activation': 'relu', 'model__alpha': ..."
2,ExtraTrees,winsorize_1_99,0.740053,80.776305,40.902793,"{'model__max_depth': 10, 'model__min_samples_l..."
3,Ridge,winsorize_1_99,0.644066,94.520608,54.046212,{'model__alpha': 30.0}


In [23]:
# Export production model predictions vs actual
prod_external_pred = production_model.predict(X_external)

export_pred_vs_actual(
    y_true=np.asarray(y_external),
    y_pred=np.asarray(prod_external_pred),
    title=f'Production External Pred vs Actual | {BEST_MODEL}_{BEST_OUTLIER}',
    out_path=VIS_DIR / 'production_external_pred_vs_actual.png',
)

export_residual_plot(
    y_true=np.asarray(y_external),
    y_pred=np.asarray(prod_external_pred),
    title=f'Production External Residuals | {BEST_MODEL}_{BEST_OUTLIER}',
    out_path=VIS_DIR / 'production_external_residuals.png',
)

export_shap_summary_if_available(
    pipeline=production_model,
    X_ref=X_all,
    feature_names=CONFIG['features'],
    prefix='production_model_train',
)

prod_pred_df = external_clean.copy()
prod_pred_df['Predicted_Production'] = prod_external_pred
prod_pred_df.to_csv(VIS_DIR / 'production_external_predictions.csv', index=False)

print('Saved visual exports and production predictions to', VIS_DIR)


  0%|          | 0/200 [00:00<?, ?it/s]

SHAP exports created for production_model_train
Saved visual exports and production predictions to /Users/rowanterra/Desktop/DiGiTerra/docs/notebook_visual_exports
